In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import when, col

# ============================================
# Read Excel File
# ============================================

df = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(
        "abfss://OOPBI@onelake.dfs.fabric.microsoft.com/lakehouse.Lakehouse/Files/data/test.xlsx"
    )

# ============================================
# Handle Missing Values
# ============================================

# Average Age
avg_age = df.select(F.avg("Age")).collect()[0][0]

# Average Salary
avg_salary = df.select(F.avg("Salary")).collect()[0][0]

df = (
    df
    .fillna({"Gender": "Not Define"})
    .fillna({"City": "Not Define"})
    .fillna({"Education": "Not Define"})
    .fillna({"Age": avg_age})
    .fillna({"Salary": avg_salary})
)

# ============================================
# Create Age Range
# ============================================

df = df.withColumn(
    "Age_Range",
    when((col("Age") >= 20) & (col("Age") <= 23), "20-23")
    .when((col("Age") > 23) & (col("Age") <= 26), "23-26")
    .when((col("Age") > 26) & (col("Age") <= 29), "26-29")
    .when((col("Age") > 29) & (col("Age") <= 32), "29-32")
    .when((col("Age") > 32) & (col("Age") <= 35), "32-35")
    .otherwise("Other")
)

# ============================================
# Create PersonID
# ============================================

df = df.withColumn(
    "PersonID",
    F.monotonically_increasing_id() + 1
)

# Move PersonID to first column

cols = ["PersonID"] + [c for c in df.columns if c != "PersonID"]

df = df.select(cols)

# ============================================
# Check Duplicates
# ============================================

duplicates = (
    df.groupBy("Name")
      .count()
      .filter(col("count") > 1)
)

display(duplicates)

# ============================================
# Save Customer Table
# ============================================

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Tables/customer")

# ============================================
# Read ISO Code File
# ============================================

iso_df = (
    spark.read
    .option("header", "true")
    .csv(
        "abfss://OOPBI@onelake.dfs.fabric.microsoft.com/lakehouse.Lakehouse/Files/data/ISO_Code.csv"
    )
)

# Remove Null Rows

iso_df = iso_df.na.drop()

display(iso_df)

# ============================================
# Save ISO Table
# ============================================

iso_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Tables/iso")
```
